# 01 · Four Parts

**RAG is not a model. It is four plain parts wired in series:**
`chunk -> embed -> retrieve -> generate`.

RAG does not teach the model anything; it pastes retrieved text into the prompt at query
time and asks the model to answer from it. In this notebook we build the four parts as four
separate functions, wire them into one chain, and answer a question end to end on CPU.

> Thesis for the whole series: these four parts are **separable** and each **caps** the
> next, so the way to understand or debug RAG is to measure them one at a time — not to
> treat the whole thing as a model.


In [1]:
import os, sys, json
from pathlib import Path

# Run from the repo root so outputs/ and data/ resolve no matter where the notebook starts.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import support
from src.config import load_settings
from src.datasets.corpus import build_corpus
from src.pipeline.embed import build_embedder

settings = load_settings()
corpus = build_corpus()
embedder = build_embedder(settings.embed)
generator, mode = support.select_generator(corpus, settings)
print("generator mode:", mode)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

generator mode: ollama:qwen2.5:0.5b


### Part 1 — chunk: split a document into passages

In [2]:
from src.pipeline.chunk import make_chunk_fn

chunk_fn = make_chunk_fn(settings.chunk.strategy, size=settings.chunk.size,
                         overlap=settings.chunk.overlap, encoder=embedder)
doc = corpus.documents[2]
print(doc.doc_id, "->", chunk_fn(doc.text))


doc03 -> ['The island of Myrelink is famous for its windmills. The tallest windmill, called the Graywing, stands 62 metres high.']


### Part 2 — embed: turn chunks and the query into vectors

In [3]:
from src.datasets.corpus import chunk_corpus

chunks, chunk_doc_ids = chunk_corpus(corpus, chunk_fn)
vectors = embedder.encode(chunks)
print("embedded", len(chunks), "chunks ->", vectors.shape)


embedded 16 chunks -> (16, 384)


### Part 3 — retrieve: cosine top-k over the chunk vectors

In [4]:
from src.pipeline.retrieve import build_retriever

question = corpus.questions_by_kind("answerable")[0]
retriever = build_retriever(settings.retrieve.method, embedder, settings.retrieve.hybrid_alpha)
retriever.index(chunks)
hits = retriever.retrieve(question.text, settings.retrieve.top_k)
print("Q:", question.text)
for h in hits:
    print(f"  {h.score:.3f}  {h.text}")


Q: What is the capital city of Volara?
  0.596  The island of Volara is the administrative centre of the Zephyr Archipelago. Its capital city is Thornmouth, which hosts the Archipelago Assembly.
  0.566  Volara has a population of 48,000 residents and covers an area of 310 square kilometres.
  0.164  Education in the archipelago is free until the age of 18. The oldest school, Thornmouth Academy, was founded in 1842.
  0.154  Pelagos is the only island in the archipelago with an active volcano, Mount Sear, which last erupted in 1911.
  0.149  The island of Suncrest grows most of the archipelago's citrus. Its orchards produce about 9,000 tonnes of oranges each year.


### Part 4 — generate: answer from the retrieved context only

In [5]:
from src.pipeline.generate import build_prompt

prompt = build_prompt(question.text, [h.text for h in hits])
print(generator.generate(prompt))


The capital city of Volara is Thornmouth.


### The chain — the four parts as one working system

In [6]:
from src.pipeline.rag import RagPipeline

rag = RagPipeline(retriever=retriever, generator=generator, settings=settings, abstain=True)
rag.index(chunks)
result = rag.answer(question.text)
print("Q:", result.question)
print("A:", result.answer)
print("gold:", question.answer, "| groundedness:", round(result.groundedness, 2))


Q: What is the capital city of Volara?
A: The capital city of Volara is Thornmouth.
gold: Thornmouth | groundedness: 1.0


### Save artifacts (figure → PNG, result → JSON)

In [7]:
from src.visualisation import bar_chart

fig = bar_chart([f"#{i+1}" for i in range(len(hits))], [h.score for h in hits],
                title=f"Retrieval scores — {question.text}", ylabel="cosine score")
fig_path = support.save_fig(fig, "01_retrieval_scores")

record = {
    "mode": mode,
    "question": result.question,
    "gold_answer": question.answer,
    "answer": result.answer,
    "groundedness": result.groundedness,
    "retrieved": [{"doc": chunk_doc_ids[h.index], "score": h.score, "text": h.text} for h in hits],
    "prompt": result.prompt,
}
json_path = support.save_json(record, "01_four_parts")
print("saved", fig_path, "and", json_path)
fig


saved outputs/figures/01_retrieval_scores.png and outputs/results/01_four_parts.json


<Figure size 700x400 with 1 Axes>